In [0]:
import os , sys

current = os.getcwd()
reusable = os.path.join(current , ".." , "..")
print(reusable)
sys.path.append(reusable)
from utils.modules import *


In [0]:
%fs ls "abfss://bronze@spotifyproject.dfs.core.windows.net/FactStream/"

In [0]:
df_factStream = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@spotifyproject.dfs.core.windows.net/FactStream/checkpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@spotifyproject.dfs.core.windows.net/FactStream/*")

In [0]:
obj = Transform()

In [0]:
df_transformed = obj.dropColumn(df_factStream,["_rescued_data"])

In [0]:
df_transformed.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@spotifyproject.dfs.core.windows.net/FactStream/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@spotifyproject.dfs.core.windows.net/FactStream/data")\
            .toTable("spotify_catalog.silver.FactStream")

In [0]:
%sql
select * from spotify_catalog.silver.factstream